In [112]:
import pandas as pd
import geopandas as gpd
import numpy as np
from scipy.spatial import cKDTree
import re
import os
import time
import requests
from dotenv import load_dotenv

Data Validation Utility

In [61]:
def validate_geodataframe(gdf, name="GeoDataFrame"):
    """
    Validate a GeoDataFrame before analysis.
    Checks for null geometries, invalid geometries, and CRS information.
    Optionally auto-fixes invalid geometries using buffer(0).
    
    Parameters:
    -----------
    gdf : geopandas.GeoDataFrame
        The GeoDataFrame to validate
    name : str
        A descriptive name for the dataset (used in print output)
    
    Returns:
    --------
    geopandas.GeoDataFrame
        The validated (and potentially fixed) GeoDataFrame
    """
    print(f"Validating '{name}'...")
    print(f"  - Total records: {len(gdf)}")
    print(f"  - CRS: {gdf.crs}")
    
    null_count = gdf.geometry.isnull().sum()
    invalid_count = (~gdf.geometry.is_valid).sum()
    
    print(f"  - Null geometries: {null_count}")
    print(f"  - Invalid geometries: {invalid_count}")
    
    if null_count > 0:
        print(f"  WARNING: Contains {null_count} null geometries - these rows may cause issues in spatial operations")
    
    if invalid_count > 0:
        print(f"  WARNING: Contains {invalid_count} invalid geometries - attempting to fix with buffer(0)...")
        gdf = gdf.copy()
        gdf['geometry'] = gdf.geometry.buffer(0)
        new_invalid_count = (~gdf.geometry.is_valid).sum()
        if new_invalid_count == 0:
            print(f"  SUCCESS: All geometries are now valid")
        else:
            print(f"  PARTIAL FIX: {new_invalid_count} geometries still invalid")
    
    print(f"  Validation complete.\n")
    return gdf

### 1. HDB resale price with coordinates

1.1 HDB resale price 2025

In [62]:
resale_price_df =  pd.read_csv("../datasets/Resale flat prices based on registration date from Jan-2017 onwards.csv")

In [63]:
def clean_street_name(street):
    if pd.isna(street): return street
    street = street.upper().strip()
    
    # Define common Singapore address abbreviations
    replacements = {
        r'\bBT\b': 'BUKIT',
        r'\bRD\b': 'ROAD',
        r'\bAVE\b': 'AVENUE',
        r'\bDR\b': 'DRIVE',
        r'\bCL\b': 'CLOSE',
        r'\bCTRL\b': 'CENTRAL',
        r'\bCPD\b': 'COMPOUND',
        r'\bCRES\b': 'CRESCENT',
        r'\bGRV\b': 'GROVE',
        r'\bHWY\b': 'HIGHWAY',
        r'\bJLN\b': 'JALAN',
        r'\bKG\b': 'KAMPONG',
        r'\bLN\b': 'LANE', 
        r'\bMT\b': 'MOUNT',
        r'\bPL\b': 'PLACE',
        r'\bRS\b': 'RISE',
        r'\bTER\b': 'TERRACE',
        r'\bTG\b': 'TANJONG',
        r'\bUPP\b': 'UPPER',
        r'\bNTH\b': 'NORTH',
        r'\bSTH\b': 'SOUTH',
        r'\bPK\b': 'PARK',
        r'\bMKT\b': 'MARKET',
        r"\bC'WEALTH\b": 'COMMONWEALTH',
        r"\bLOR\b": 'LORONG',
        r'\bHTS\b': 'HEIGHTS',
        r'\bGDNS\b': 'GARDENS',
        r'\bST\b(?!\.)': 'STREET'
    }
    
    for short, full in replacements.items():
        street = re.sub(short, full, street)
    
    # Remove extra spaces caused by regex
    return " ".join(street.split())

In [64]:
# filter only 2019-2025 resale data 
resale_price_df["month"]= pd.to_datetime(resale_price_df["month"])
resale_df = resale_price_df[(resale_price_df["month"].dt.year >= 2019) & (resale_price_df["month"].dt.year <= 2025)].copy()
resale_df['street_name'] = resale_df['street_name'].apply(clean_street_name)

# create year and month columns 
resale_df['year'] = resale_df['month'].dt.year
resale_df['month'] = resale_df['month'].dt.month
resale_df.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,year
42070,1,ANG MO KIO,3 ROOM,225,ANG MO KIO AVENUE 1,01 TO 03,67.0,New Generation,1978,58 years,230000.0,2019
42071,1,ANG MO KIO,3 ROOM,174,ANG MO KIO AVENUE 4,01 TO 03,60.0,Improved,1986,66 years,235000.0,2019
42072,1,ANG MO KIO,3 ROOM,440,ANG MO KIO AVENUE 10,04 TO 06,67.0,New Generation,1979,59 years,238000.0,2019
42073,1,ANG MO KIO,3 ROOM,174,ANG MO KIO AVENUE 4,10 TO 12,61.0,Improved,1986,66 years 01 month,240000.0,2019
42074,1,ANG MO KIO,3 ROOM,637,ANG MO KIO AVENUE 6,01 TO 03,68.0,New Generation,1980,60 years 08 months,240000.0,2019


1.2 Existing HDBs

In [140]:
# Instead of scraping from OneMap, we can use the HDB Existing Building dataset which contains the coordinates for all HDB blocks. 
# We will match on block and street name to get the coordinates for each resale transaction.
hdb_existing = gpd.read_file("../datasets/HDBExistingBuilding.geojson")
road_namecode = pd.read_excel("../datasets/road_name_code.xlsx", skiprows= 10)

In [141]:
road_namecode = road_namecode[["Domain Value", "Description"]].copy()

# match on "ST_COD" = "Domain Value" to get road description
hdb_existing = hdb_existing.merge(
    road_namecode,
    left_on= "ST_COD", 
    right_on= "Domain Value", 
    how= "left"
)
# ensure all ST_COD has a match
unmatched_rows = hdb_existing[hdb_existing['Domain Value'].isna()]
if unmatched_rows.empty:
    print("All 'ST_COD' values found a match in the road name table.")
else:
    print(f"{len(unmatched_rows)} rows failed to match.")
    print("Missing ST_COD values:")
    print(unmatched_rows['ST_COD'].unique())
    
# rename and select columns
hdb_existing = hdb_existing.rename(columns={"Description": "STREET_NAME", "geometry": "GEOMETRY"})
hdb_existing = hdb_existing[["BLK_NO", "STREET_NAME", "GEOMETRY", "POSTAL_COD"]]
hdb_existing.head()

All 'ST_COD' values found a match in the road name table.


,BLK_NO,STREET_NAME,GEOMETRY,POSTAL_COD
0,208B,CLEMENTI AVENUE 6,"POLYGON ((103.76235 1.32274, 103.76232 1.32274...",122208
1,238,BUKIT BATOK EAST AVENUE 5,"POLYGON ((103.75446 1.3501, 103.75448 1.35011,...",650238
2,88,CIRCUIT ROAD,"POLYGON ((103.88589 1.32371, 103.88579 1.3237,...",370088
3,44A,MARINE CRESCENT,"POLYGON ((103.91325 1.30501, 103.91325 1.30503...",441044
4,671B,JURONG WEST STREET 65,"POLYGON ((103.70208 1.34397, 103.70208 1.34398...",642671


1.3 Merge existing HDBs geometry with resale prices

In [142]:
# standardize to string types
resale_df['block'] = resale_df['block'].astype(str).str.strip()
hdb_existing['BLK_NO'] = hdb_existing['BLK_NO'].astype(str).str.strip()
# match on block and street name to get coordinates
resale_with_geom_df = resale_df.merge(
    hdb_existing, 
    left_on=['block', 'street_name'], 
    right_on=['BLK_NO', 'STREET_NAME'], 
    how='left'
)
# rename and select columns
resale_with_geom_df = resale_with_geom_df.rename(columns={'GEOMETRY': 'geometry', 'POSTAL_COD': 'postal_code'})
selected_columns = [
    'month', 
    'year',
    'town', 
    'flat_type', 
    'block', 
    'street_name', 
    'storey_range', 
    'floor_area_sqm', 
    'flat_model', 
    'lease_commence_date', 
    'remaining_lease', 
    'resale_price', 
    'geometry',
    'postal_code'
]
resale_with_geom_df = resale_with_geom_df[selected_columns]
# add address_cleaned column by concatenating block, street name
resale_with_geom_df.head()
resale_with_geom_df['address_clean'] = resale_with_geom_df['block'] + " "+ resale_with_geom_df['street_name']

In [163]:
# convert dataframe to geodataframe
resale_gdf = gpd.GeoDataFrame(resale_with_geom_df, geometry='geometry')
# reproject from geographic CRS (WGS84) to a projected CRS using EPSG:3414 (SVY21)
resale_gdf = resale_gdf.to_crs(3414)
# convert hdb polygons to points (centroids)
resale_gdf['geometry'] = resale_gdf['geometry'].centroid
resale_gdf.to_file("resale_2019_to_2025_cleaned.geojson", driver="GeoJSON")
resale_gdf.head(5)

,month,year,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,geometry,postal_code,address_clean
0,1,2019,ANG MO KIO,3 ROOM,225,ANG MO KIO AVENUE 1,01 TO 03,67.0,New Generation,1978,58 years,230000.0,POINT (28537.68 38825.233),560225,225 ANG MO KIO AVENUE 1
1,1,2019,ANG MO KIO,3 ROOM,174,ANG MO KIO AVENUE 4,01 TO 03,60.0,Improved,1986,66 years,235000.0,POINT (28487.136 39687.765),560174,174 ANG MO KIO AVENUE 4
2,1,2019,ANG MO KIO,3 ROOM,440,ANG MO KIO AVENUE 10,04 TO 06,67.0,New Generation,1979,59 years,238000.0,POINT (30336.26 38718.18),560440,440 ANG MO KIO AVENUE 10
3,1,2019,ANG MO KIO,3 ROOM,174,ANG MO KIO AVENUE 4,10 TO 12,61.0,Improved,1986,66 years 01 month,240000.0,POINT (28487.136 39687.765),560174,174 ANG MO KIO AVENUE 4
4,1,2019,ANG MO KIO,3 ROOM,637,ANG MO KIO AVENUE 6,01 TO 03,68.0,New Generation,1980,60 years 08 months,240000.0,POINT (29003.603 40255.313),560637,637 ANG MO KIO AVENUE 6


### 2. Get school coordinates using OneMap API

In [ ]:
## Get centroid coordinates for schools using OneMap API
df = pd.read_csv("../school_scoring/schools_tiered.csv")

# output columns
for col in ["latitude", "longitude", "searchval", "formatted_address", "postal_found", "geocode_status", "matched_query"]:
    if col not in df.columns:
        df[col] = None

# import request
base_url = "https://www.onemap.gov.sg/api/common/elastic/search"

count = 0

for i in range(len(df)):
    if pd.notna(df.loc[i, "latitude"]) and pd.notna(df.loc[i, "longitude"]):
        print(f"Row {i}: already geocoded")
        continue

    school_name = str(df.loc[i, "school_name"]).strip() if pd.notna(df.loc[i, "school_name"]) else ""
    address = str(df.loc[i, "address"]).strip() if pd.notna(df.loc[i, "address"]) else ""
    postal_code = str(df.loc[i, "postal_code"]).strip() if pd.notna(df.loc[i, "postal_code"]) else ""

    queries = []

    if postal_code and postal_code.lower() != "nan":
        queries.append(("postal_code", postal_code))

    if address and address.lower() != "nan":
        queries.append(("address", f"{address} Singapore"))

    if school_name and school_name.lower() != "nan":
        queries.append(("school_name", school_name))

    result = None
    matched_query_type = None

    try:
        for query_type, q in queries:
            params = {
                "searchVal": q,
                "returnGeom": "Y",
                "getAddrDetails": "Y",
                "pageNum": 1
            }

            resp = requests.get(base_url, params=params, timeout=15)
            time.sleep(0.5)
            resp.raise_for_status()
            data = resp.json()

            if data.get("results") and len(data["results"]) > 0:
                result = data["results"][0]
                matched_query_type = query_type
                break

        if result is not None:
            df.loc[i, "longitude"] = result.get("LONGITUDE")
            df.loc[i, "latitude"] = result.get("LATITUDE")
            df.loc[i, "searchval"] = result.get("SEARCHVAL")
            df.loc[i, "formatted_address"] = result.get("ADDRESS")
            df.loc[i, "postal_found"] = result.get("POSTAL")
            df.loc[i, "geocode_status"] = "success"
            df.loc[i, "matched_query"] = matched_query_type

            print(
                f"{school_name}: success via {matched_query_type} | "
                f"lat={result.get('LATITUDE')}, lon={result.get('LONGITUDE')}"
            )
        else:
            df.loc[i, "geocode_status"] = "not_found"
            df.loc[i, "matched_query"] = None
            print(f"{school_name}: not found")

    except Exception as e:
        df.loc[i, "geocode_status"] = f"error: {str(e)}"
        df.loc[i, "matched_query"] = None
        print(f"{school_name}: error -> {e}")

    count += 1

    if count % 50 == 0:
        df.to_csv("schools_geocoded_partial.csv", index=False)
        print(f"Saved partial progress at row {i}. Sleeping for 2 seconds...")
        time.sleep(2)

# df.to_csv("schools_geocoded.csv", index=False)
schools_gdf = df
print("Done.")


NANYANG PRIMARY SCHOOL: success via postal_code | lat=1.321070942411543, lon=103.8076818527984
CATHOLIC HIGH SCHOOL: success via postal_code | lat=1.354525170657562, lon=103.8449008048039
CHIJ ST. NICHOLAS GIRLS' SCHOOL: success via postal_code | lat=1.373476878789176, lon=103.8342532694353
TAO NAN SCHOOL: success via postal_code | lat=1.305285286873217, lon=103.9115531523253
NAN HUA PRIMARY SCHOOL: success via postal_code | lat=1.319201599563877, lon=103.7610950657604
AI TONG SCHOOL: success via postal_code | lat=1.360583433890406, lon=103.8330203339857
ANGLO-CHINESE SCHOOL (PRIMARY): success via postal_code | lat=1.31837054523522, lon=103.8356097323541
MARIS STELLA HIGH SCHOOL: success via postal_code | lat=1.34139181236128, lon=103.8777407134485
ROSYTH SCHOOL: success via postal_code | lat=1.372858365405815, lon=103.8747717039899
HOLY INNOCENTS' PRIMARY SCHOOL: success via postal_code | lat=1.366938308773486, lon=103.8941148997942
PEI HWA PRESBYTERIAN PRIMARY SCHOOL: success via pos

In [150]:
# verify schools have valid geocode
print(schools_gdf[schools_gdf["geocode_status"] != "success"])

# reproject from geographic CRS (WGS84) to a projected CRS using EPSG:3414 (SVY21)
schools_gdf = gpd.GeoDataFrame(
    schools_gdf,
    geometry=gpd.points_from_xy(schools_gdf["longitude"], schools_gdf["latitude"]),
    crs="EPSG:4326"
)
schools_gdf = schools_gdf.to_crs(epsg=3414)

Empty GeoDataFrame
Columns: [school_name, address, postal_code, nature_code, sap_ind, autonomous_ind, gifted_ind, P1_demand, P2A_demand, P2B_demand, P2C_demand, P2CS_demand, subject_count, distprog_count, cca_clubs, cca_others, cca_sports, cca_uniformed, cca_arts, affiliation_count, top_psle_score, school_score_raw, school_score, school_tier, latitude, longitude, searchval, formatted_address, postal_found, geocode_status, matched_query, geometry]
Index: []

[0 rows x 32 columns]


### 3. Get feature variables of schools

3.1 
- num_schools_1km
- num_schools_2km

In [144]:
# get unique hdb locations
unique_hdb = resale_gdf.drop_duplicates(subset=["postal_code"]).copy()
key_col = "postal_code"

# prepare unique hdb and school coordinates for distance calculation
school_coords = np.array([(geom.x, geom.y) for geom in schools_gdf.geometry])
hdb_coords = np.array([(geom.x, geom.y) for geom in unique_hdb.geometry])

school_names = schools_gdf["school_name"].tolist()

# build cKDTree for efficient spatial queries
tree = cKDTree(school_coords)

# get schools within 1km and 2km
indices_1km = tree.query_ball_point(hdb_coords, r=1000)
indices_2km = tree.query_ball_point(hdb_coords, r=2000)

# convert index arrays into school name lists
unique_hdb["schools_within_1km"] = [
    [school_names[i] for i in idx_list] for idx_list in indices_1km]

unique_hdb["schools_within_2km"] = [
    [school_names[i] for i in idx_list] for idx_list in indices_2km]

# count columns
unique_hdb["num_schools_1km"] = unique_hdb["schools_within_1km"].apply(len)
unique_hdb["num_schools_2km"] = unique_hdb["schools_within_2km"].apply(len)

3.2 
- num_tier1_schools_1km
- num_tier1_schools_2km

In [145]:
tier1 = schools_gdf[schools_gdf["school_tier"] == "Tier 1"].copy()
tier1_coords = np.array([(geom.x, geom.y) for geom in tier1.geometry])
tier1_names = tier1["school_name"].tolist()

tier1_tree = cKDTree(tier1_coords)

tier1_idx_1km = tier1_tree.query_ball_point(hdb_coords, r=1000)
tier1_idx_2km = tier1_tree.query_ball_point(hdb_coords, r=2000)

unique_hdb["tier1_schools_within_1km"] = [
    [tier1_names[i] for i in idx_list] for idx_list in tier1_idx_1km]

unique_hdb["tier1_schools_within_2km"] = [
    [tier1_names[i] for i in idx_list] for idx_list in tier1_idx_2km]

unique_hdb["num_tier1_schools_1km"] = unique_hdb["tier1_schools_within_1km"].apply(len)
unique_hdb["num_tier1_schools_2km"] = unique_hdb["tier1_schools_within_2km"].apply(len)

3.3
- num_tier2_schools_1km
- num_tier2_schools_2km

In [146]:
tier2 = schools_gdf[schools_gdf["school_tier"] == "Tier 2"].copy()
tier2_coords = np.array([(geom.x, geom.y) for geom in tier2.geometry])
tier2_names = tier2["school_name"].tolist()

tier2_tree = cKDTree(tier2_coords)

tier2_idx_1km = tier2_tree.query_ball_point(hdb_coords, r=1000)
tier2_idx_2km = tier2_tree.query_ball_point(hdb_coords, r=2000)

unique_hdb["tier2_schools_within_1km"] = [
    [tier2_names[i] for i in idx_list] for idx_list in tier2_idx_1km]

unique_hdb["tier2_schools_within_2km"] = [
    [tier2_names[i] for i in idx_list] for idx_list in tier2_idx_2km]

unique_hdb["num_tier2_schools_1km"] = unique_hdb["tier2_schools_within_1km"].apply(len)
unique_hdb["num_tier2_schools_2km"] = unique_hdb["tier2_schools_within_2km"].apply(len)

In [156]:
# merge back to full resale dataset if needed
school_feature_test = unique_hdb[[
    "postal_code",
    "num_schools_1km",
    "schools_within_1km",
    "num_schools_2km",
    "schools_within_2km",
    "num_tier1_schools_1km",
    "num_tier1_schools_2km",
    "num_tier2_schools_1km",
    "num_tier2_schools_2km"
]].copy()

resale_with_school_test = resale_gdf.merge(
    school_feature_test,
    on="postal_code",
    how="left"
)

resale_with_school_test.to_csv("resale_with_school_test.csv", index=False)


In [ ]:
# merge back to full resale dataset if needed
school_features = unique_hdb[[
    "postal_code",
    "num_schools_1km",
    "num_schools_2km",
    "num_tier1_schools_1km",
    "num_tier1_schools_2km",
    "num_tier2_schools_1km",
    "num_tier2_schools_2km"
]].copy()

resale_with_school = resale_gdf.merge(
    school_features,
    on="postal_code",
    how="left"
)
resale_with_school.head(5)


### 4. Check data integrity 

In [ ]:
# define logical conditions to check for data integrity
mask = (
    (resale_with_school['num_tier1_schools_1km'] > resale_with_school['num_schools_1km']) |
    (resale_with_school['num_tier1_schools_2km'] > resale_with_school['num_schools_2km']) |
    (resale_with_school['num_tier2_schools_1km'] > resale_with_school['num_schools_1km']) |
    (resale_with_school['num_tier2_schools_2km'] > resale_with_school['num_schools_2km'])
)

# check if any row returned True for the errors
if mask.any():
    error_count = mask.sum()
    raise ValueError(f"Data Integrity Error: {error_count} records have tier counts exceeding total school counts.")
else:
    resale_with_school.to_csv("resale_with_school.csv", index=False)
    print("Validation passed. File saved successfully.")

Validation passed. File saved successfully.
